# LD-pruned biallelic SNP calls

LD pruning removes redundant SNPs that are in linkage disequilibrium, reducing correlation between variants. This is important for analyses like PCA and ADMIXTURE where independent markers are assumed.

`biallelic_snp_calls_ld_pruned()` wraps `biallelic_snp_calls()` and applies LD pruning via `scikit-allel`'s `locate_unlinked()`. The output retains the same dataset structure and is compatible with existing downstream methods.

## Setup

In [ ]:
import malariagen_data

In [ ]:
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../gcs_cache"),
    results_cache="results_cache",
)
ag3

## Basic usage

In [ ]:
ds_pruned = ag3.biallelic_snp_calls_ld_pruned(
    region="3L",
    n_snps=100_000,
    sample_sets="AG1000G-BF-A",
)
ds_pruned

In [ ]:
# Inspect dimensions.
print(f"variants: {ds_pruned.sizes['variants']}")
print(f"samples:  {ds_pruned.sizes['samples']}")
print(f"alleles:  {ds_pruned.sizes['alleles']}")

## Effect of LD parameters

In [ ]:
# Stricter threshold (0.1, default) removes more correlated SNPs.
ds_strict = ag3.biallelic_snp_calls_ld_pruned(
    region="3L",
    n_snps=100_000,
    sample_sets="AG1000G-BF-A",
    ld_threshold=0.1,
)

# More lenient threshold retains more SNPs.
ds_lenient = ag3.biallelic_snp_calls_ld_pruned(
    region="3L",
    n_snps=100_000,
    sample_sets="AG1000G-BF-A",
    ld_threshold=0.5,
)

print(f"strict  (threshold=0.1): {ds_strict.sizes['variants']} variants")
print(f"lenient (threshold=0.5): {ds_lenient.sizes['variants']} variants")

## Before vs after pruning

In [ ]:
# Get the thinned (but not LD-pruned) SNPs for comparison.
ds_before = ag3.biallelic_snp_calls(
    region="3L",
    n_snps=100_000,
    sample_sets="AG1000G-BF-A",
)

print(f"before LD pruning: {ds_before.sizes['variants']} variants")
print(f"after  LD pruning: {ds_pruned.sizes['variants']} variants")
print(f"removed:           {ds_before.sizes['variants'] - ds_pruned.sizes['variants']} variants")

## Downstream compatibility

The pruned dataset retains the same structure as `biallelic_snp_calls()` output, so it can be passed directly into existing workflows like PCA.

In [ ]:
# Verify the pruned dataset has all variables expected by downstream methods.
assert "call_genotype" in ds_pruned
assert "variant_allele" in ds_pruned
assert "variant_contig" in ds_pruned.coords
assert "variant_position" in ds_pruned.coords
assert "sample_id" in ds_pruned.coords

# Shape sanity check.
n_variants = ds_pruned.sizes["variants"]
n_samples = ds_pruned.sizes["samples"]
assert ds_pruned["call_genotype"].shape == (n_variants, n_samples, 2)
assert ds_pruned["variant_allele"].shape == (n_variants, 2)

print(f"Dataset is valid: {n_variants} variants × {n_samples} samples")